# Using Lacuna with Docker

This guide demonstrates how to run Lacuna analyses using the pre-built Docker image, without installing Lacuna or its dependencies locally. We run a structural network mapping (SNM) analysis using the HCP1065 tractogram as a complete example.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/how-to/docker.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

## Prerequisites

You need Docker installed on your system. See [Get Docker](https://docs.docker.com/get-docker/) for installation instructions.

The Lacuna Docker image includes all dependencies (Python, MRtrix3, TemplateFlow templates) so you do not need to install anything else.

## Setup

Pull the Lacuna Docker image.

In [ ]:
!docker pull ghcr.io/m-petersen/lacuna:edge

edge: Pulling from m-petersen/lacuna

6da507ff: Pulling fs layer 
007a5b45: Pulling fs layer 
92e2619d: Pulling fs layer 
aba74b8c: Pulling fs layer 
7d28006b: Pulling fs layer 
a052e37e: Pulling fs layer 
db6f03a5: Pulling fs layer 
7d28006b: Waiting fs layer 
db6f03a5: Waiting fs layer 
4c24b1f6: Pulling fs layer 
aba74b8c: Waiting fs layer 
Digest: sha256:69c83a8ec89f0325cba5b8d93eca542a89ed37c3441331fb2ac5f6343e7b86a0
Status: Downloaded newer image for ghcr.io/m-petersen/lacuna:edge
ghcr.io/m-petersen/lacuna:edge


Verify the image works.

In [ ]:
!docker run --rm ghcr.io/m-petersen/lacuna:edge --help

usage: lacuna [-h] [--version] <command> ...

Lacuna v0.0.1.dev383

options:
  -h, --help  show this help message and exit
  --version   show program's version number and exit

commands:
  Use 'lacuna <command> --help' for more information.

  <command>
    fetch     Download and setup connectomes
    run       Run lesion network mapping analyses
    collect   Aggregate parcelstats across subjects
    info      Display available resources (atlases, connectomes)
    bidsify   Convert NIfTI files to BIDS format
    tutorial  Setup tutorial data for learning Lacuna

Commands:
  bidsify   Convert NIfTI files to BIDS format
  fetch     Download and setup connectomes for analysis
  run       Run lesion analyses
  collect   Aggregate results across subjects
  info      Display available resources (atlases, connectomes)
  tutorial  Setup tutorial data for learning Lacuna

Examples:
  lacuna tutorial ./my_tutorial
  lacuna fetch gsp1000 --api-key \$DATAVERSE_API_KEY
  lacuna run rd /bids /outpu

## How volume mounts work

Docker containers are isolated from the host filesystem. To give the container access to your data, you mount host directories into the container using `-v`:

```
-v /host/path:/container/path:ro   # read-only
-v /host/path:/container/path      # read-write
```

| Purpose | Host path | Container path | Mode |
|---------|-----------|----------------|------|
| BIDS input data | `/path/to/bids` | `/bids` | read-only (`:ro`) |
| Output results | `/path/to/output` | `/output` | read-write |
| Connectomes | `/path/to/connectomes` | `/connectomes` | read-only (`:ro`) |

The container entrypoint is the `lacuna` command, so you pass subcommands directly after the image name.

## Prepare tutorial data

We use Lacuna inside the container to create a tutorial dataset. The `--rm` flag removes the container after it exits.

In [9]:
!mkdir -p /tmp/docker_tutorial
!docker run --rm -u $(id -u):$(id -g)\
    -v /tmp/docker_tutorial:/data \
    ghcr.io/m-petersen/lacuna:edge \
    tutorial /data/bids --force


Setting up tutorial data at: /data/bids
✓ Tutorial data copied to: /data/bids

The tutorial dataset includes:
  - 3 synthetic subjects (sub-01, sub-02, sub-03)
  - Binary lesion masks in MNI152NLin6Asym space
  - BIDS-compliant structure


In [ ]:
!ls /tmp/docker_tutorial/bids/

## Run structural network mapping

In [10]:
!docker run --rm -u $(id -u):$(id -g)\
    -v /tmp/docker_tutorial:/data \
    ghcr.io/m-petersen/lacuna:edge \
    fetch hcp1065 --output-dir /data/connectomes

Fetching HCP1065 structural tractogram...
  Output: /data/connectomes
  Keep original: True

hcp1065_avg_tracts_trk.zip: 100%|██████████| 588M/588M [01:02<00:00, 9.39MB/s] 
Merging tracts:   0%|          | 0/77 [00:00<?, ?it/s]Found 77 tract files (10 excluded)
Loading and merging streamlines...
Merging tracts: 100%|██████████| 77/77 [00:15<00:00,  4.91it/s]
Processed 77 files, 479804 total streamlines
Saving merged tractogram to /data/connectomes/hcp1065.tck...
Merge complete: /data/connectomes/hcp1065.tck

✓ HCP1065 fetch complete!
  Files: 2
  Duration: 85.9s
  Output: /data/connectomes


In [11]:
!ls /tmp/docker_tutorial/connectomes/

hcp1065_avg_tracts_trk.zip  hcp1065.tck  hcp1065_tracts


In [ ]:
!mkdir -p /tmp/docker_tutorial/output
!docker run --rm -u $(id -u):$(id -g)\
    -v /tmp/docker_tutorial/bids:/bids:ro \
    -v /tmp/docker_tutorial/connectomes:/connectomes:ro \
    -v /tmp/docker_tutorial/output:/output \
    ghcr.io/m-petersen/lacuna:edge \
    run snm /bids /output \
    --connectome-path /connectomes/hcp1065.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym

Inspect the output. The results are written to the host filesystem via the volume mount.

In [19]:
!ls /tmp/docker_tutorial/output/sub-01/ses-01/anat/

sub-01_ses-01_desc-provenance.json
sub-01_ses-01_label-acuteinfarct_summarystatistics_stats.json
sub-01_ses-01_space-MNI152NLin2009cAsym_label-acuteinfarct_mask.nii.gz
sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_desc-snm_disconnectionmap.json
sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_desc-snm_disconnectionmap.nii.gz


## Tips

- Always mount input data as read-only (`:ro`) to prevent accidental modification.
- Mount the connectome directory separately from the BIDS input for clarity.
- Use `--rm` to automatically clean up containers after they exit.
- For HPC environments, consider using [Apptainer](https://apptainer.org/) (formerly Singularity) instead — it can pull the same Docker image with `apptainer pull docker://ghcr.io/m-petersen/lacuna:edge`.
- The container includes pre-fetched TemplateFlow templates, so no internet access is needed at runtime for spatial transformations.